# Лабораторная работа 4. Настройка среды Apache Spark и работа с объектами Spark DataFrame

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/l04.ipynb)

**Цели занятия:**

- Понять архитектуру Apache Spark: драйвер, исполнители, менеджер кластера, ленивые вычисления
- Создать `SparkSession`, прочитать CSV с автоопределением и с явной схемой
- Выполнять агрегации DataFrame API и запросы Spark SQL
- Посчитать частоты слов узбекского корпуса через RDD и через DataFrame и сохранить результат в Parquet

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"

# Для заданий по Spark нужен настоящий PySpark (в Colab ставится за ~1 минуту)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Что такое Apache Spark

**Apache Spark** — универсальный движок для распределённой обработки больших данных. В отличие от MapReduce, который после каждого шага пишет промежуточные данные на диск, Spark держит их **в оперативной памяти**, поэтому многошаговые и итеративные задачи (машинное обучение, графы, интерактивный анализ) выполняются в десятки раз быстрее.

| Модуль | Назначение |
|---|---|
| Spark Core (RDD) | базовые распределённые коллекции и планировщик задач |
| **Spark SQL / DataFrame** | таблицы со схемой, SQL-запросы, оптимизатор Catalyst |
| Structured Streaming | обработка потоков данных (Kafka, файлы, сокеты) |
| MLlib | машинное обучение: классификация, кластеризация, TF-IDF, Word2Vec |
| GraphX | графовые алгоритмы |

Программы пишут на Python (**PySpark**), Scala, Java, R или SQL. Spark читает данные из HDFS, S3, баз данных, локальных файлов и может работать поверх YARN.

## Архитектура

```text
 +--------------- Driver ----------------+        +------------ Executor 1 ----+
 | ваша программа + SparkSession         | -----> | task  task  [кэш данных]   |
 | строит план (DAG), делит его на этапы |        +----------------------------+
 | (stages) и задачи (tasks)             |        +------------ Executor 2 ----+
 +---------------+-----------------------+ -----> | task  task  [кэш данных]   |
                 |                                +----------------------------+
                 | запрос ресурсов
                 v
      Cluster manager: local / Standalone / YARN / Kubernetes
      (выделяет ресурсы и запускает исполнителей на узлах)
```

- **Driver** — процесс, в котором работает ваш код и `SparkSession`; он превращает операции в план и раздаёт задачи.
- **Executors** — процессы на узлах кластера, выполняющие задачи и хранящие кэш.
- **Cluster manager** — выделяет ресурсы. Адрес задаётся параметром `master`:

| master | Значение |
|---|---|
| `local` | один поток на этой машине |
| `local[4]` | 4 потока |
| `local[*]` | столько потоков, сколько ядер процессора |
| `yarn` | кластер Hadoop YARN |
| `spark://host:7077` | собственный кластер Spark (Standalone) |

## Ленивые вычисления: трансформации и действия

Операции Spark делятся на два вида:

| Трансформации (ленивые) | Действия (запускают вычисления) |
|---|---|
| `select`, `filter`, `withColumn`, `groupBy`, `join`, `orderBy` | `show`, `count`, `collect`, `take`, `first` |
| RDD: `map`, `flatMap`, `filter`, `reduceByKey` | запись: `write.parquet(...)`, `saveAsTextFile` |

Трансформация лишь **дописывает шаг в план** и мгновенно возвращает новый DataFrame. Только **действие** запускает задание (job): Spark оптимизирует весь план целиком, делит его на этапы по границам **перемешивания** (shuffle — например, `groupBy` или `reduceByKey`) и запускает по одной задаче на каждый **раздел** (partition) данных.

## RDD и DataFrame

| | RDD | DataFrame |
|---|---|---|
| Данные | произвольные объекты Python | строки со схемой (столбцы и типы) |
| API | `map`, `flatMap`, `reduceByKey` | `select`, `filter`, `groupBy`, SQL |
| Оптимизация | нет — выполняется как написано | оптимизатор Catalyst + движок Tungsten |
| Когда использовать | неструктурированный текст, нестандартная логика | таблицы, агрегации — почти всегда |

## Первые шаги в PySpark

```python
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("UzBigData")
         .master("local[*]")
         .getOrCreate())          # повторный вызов вернёт уже созданную сессию
sc = spark.sparkContext           # «старый» вход для RDD
print(spark.version, sc.master, sc.defaultParallelism)

df = spark.read.csv("data/spark/sales_events.csv", header=True, inferSchema=True)
df.printSchema()
df.groupBy("region").agg(F.sum("qty").alias("items")).orderBy(F.desc("items")).show()
```

Схему можно задать явно — это быстрее (не нужен лишний проход по данным) и надёжнее:

```python
schema = "event_id INT, event_time TIMESTAMP, region STRING, qty INT, unit_price DOUBLE"
# или StructType([StructField("event_id", IntegerType(), False), ...])
df = spark.read.csv(path, header=True, schema=schema)
```

**Spark SQL**: зарегистрируйте DataFrame как временное представление и пишите обычный SQL:

```python
df.createOrReplaceTempView("sales")
spark.sql("SELECT region, COUNT(*) AS n FROM sales GROUP BY region ORDER BY n DESC").show()
```

**Запись результатов** создаёт не файл, а **каталог** с частями `part-00000-…` (по одной на раздел) и пустым маркером `_SUCCESS`:

| mode | Если каталог уже есть |
|---|---|
| `error` / `errorifexists` (по умолчанию) | ошибка |
| `overwrite` | перезаписать |
| `append` | добавить новые части |
| `ignore` | ничего не делать |

```python
df.write.mode("overwrite").parquet("output/sales_parquet")
df.write.mode("overwrite").csv("output/sales_csv", header=True)
back = spark.read.parquet("output/sales_parquet")
```

**Parquet** — колоночный формат со сжатием и сохранённой схемой; стандарт де-факто для хранения таблиц в Big Data.

## Spark UI

Пока работает приложение, драйвер поднимает веб-интерфейс **http://localhost:4040**: вкладки *Jobs* (задания), *Stages* (этапы и задачи), *Storage* (кэш), *SQL / DataFrame* (планы запросов), *Executors*. Это главный инструмент для поиска «узких мест».

> **Заметка: Как устроен тренажёр**  
> В браузере работает **учебный эмулятор PySpark** на основе pandas: API и формат вывода (`show`, `printSchema`) повторяют PySpark 3.5, но вычисления идут сразу в одном процессе, а Spark UI нет. Активная `SparkSession` сохраняется между запусками: `getOrCreate()` возвращает уже созданную. Тот же код без изменений работает в Google Colab и на кластере.

## Задание 1. Создание SparkSession

Создайте сессию Spark с именем приложения **`UzBigData-L4`** и мастером, использующим все ядра локальной машины.

Сохраните в переменные:
- `version` — версию Spark (`spark.version`);
- `master` — адрес мастера (`spark.sparkContext.master`);
- `cores` — параллелизм по умолчанию (`spark.sparkContext.defaultParallelism`) — столько задач Spark запускает одновременно.

Выведите их на экран. Последней строкой оставьте `spark` — увидите карточку сессии.

<details><summary>💡 Подсказка</summary>

`.appName("UzBigData-L4").master("local[*]")`; версия — `spark.version`, мастер — `spark.sparkContext.master`.

</details>

In [ ]:
from pyspark.sql import SparkSession

# Если сессия уже создана (например, в другом задании), остановим её,
# иначе getOrCreate() вернёт старую сессию со старыми настройками
active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (SparkSession.builder
         .appName("...")     # TODO: имя приложения
         .master("...")      # TODO: все ядра локальной машины
         .getOrCreate())

version = ...
master = ...
cores = ...

print("Spark:", version)
print("Master:", master)
print("Параллелизм:", cores)
spark

## Задание 2. Чтение CSV и схема данных

Файл `data/spark/sales_events.csv` — 3000 продаж магазинов в регионах Узбекистана: `event_id, event_time, region, product, category, qty, unit_price, payment`.

1. Прочитайте его в DataFrame `df` так, чтобы имена столбцов взялись из первой строки, а типы определились автоматически.
2. Выведите схему (`printSchema()`) и первые 5 строк (`show(5)`).
3. Сохраните число строк в `n_rows`, число столбцов — в `n_cols`.

Сначала запустите стартовый код как есть и посмотрите, что получится без параметров.

<details><summary>💡 Подсказка</summary>

`spark.read.csv(путь, header=True, inferSchema=True)`; число строк — действие `df.count()`, столбцов — `len(df.columns)`.

</details>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("UzBigData-L4").getOrCreate()

df = spark.read.csv("data/spark/sales_events.csv")   # TODO: добавьте параметры

df.printSchema()
df.show(5)

n_rows = ...
n_cols = ...
print("Строк:", n_rows, "столбцов:", n_cols)

## Задание 3. Явная схема и новый столбец

Автоопределение типов требует лишнего прохода по данным и может ошибиться. Задайте схему явно.

1. Опишите схему `schema` для всех 8 столбцов — через `StructType` или DDL-строкой. Типы: `event_id` и `qty` — INT, `event_time` — TIMESTAMP, `unit_price` — **DOUBLE** (цены бывают дробными), остальные — STRING.
2. Прочитайте файл с этой схемой в DataFrame `sales` (`header=True, schema=schema`).
3. Добавьте столбец `revenue = qty * unit_price` (`withColumn`) — сумма продажи в сумах.
4. Выведите схему и 5 строк.

<details><summary>💡 Подсказка</summary>

Добавьте поля `StructField("region", StringType(), True)` и т.д.; новый столбец: `sales.withColumn("revenue", F.col("qty") * F.col("unit_price"))`.

</details>

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("event_id", IntegerType(), False),
    StructField("event_time", TimestampType(), True),
    # TODO: region, product, category (строки), qty (int), unit_price (double), payment (строка)
])
# Другой способ — DDL-строка:
# schema = "event_id INT, event_time TIMESTAMP, region STRING, ..."

sales = spark.read.csv("data/spark/sales_events.csv", header=True, schema=schema)
sales = sales   # TODO: добавьте столбец revenue

sales.printSchema()
sales.show(5)

## Задание 4. Агрегации: выручка по регионам и способам оплаты

DataFrame `sales` со столбцом `revenue` уже загружен (см. подготовительный код).

1. `by_region` — по каждому региону: `orders` — число продаж, `items` — сумма `qty`, `revenue` — сумма выручки. Отсортируйте по убыванию выручки.
2. `by_payment` — по каждому способу оплаты (`payment`): `orders` — число продаж и `avg_check` — средняя сумма продажи, округлённая до 2 знаков. Отсортируйте по убыванию `orders`.
3. `top_region` — **строка** с названием региона с наибольшей выручкой (возьмите из `by_region` методом `first()`).

Выведите обе таблицы методом `show()`.

<details><summary>💡 Подсказка</summary>

`sales.groupBy("region").agg(F.count("*").alias("orders"), F.sum("qty").alias("items"), F.sum("revenue").alias("revenue")).orderBy(F.desc("revenue"))`; `top_region = by_region.first()["region"]`.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
sales = (spark.read.csv("data/spark/sales_events.csv", header=True, inferSchema=True)
         .withColumn("revenue", F.col("qty") * F.col("unit_price")))

In [ ]:
from pyspark.sql import functions as F

by_region = (sales.groupBy("region")
             .agg(
                 # TODO: F.count(...).alias("orders"), F.sum(...).alias("items"), F.sum(...).alias("revenue")
             ))
by_region.show()

by_payment = ...
# by_payment.show()

top_region = ...
print("Лидер по выручке:", top_region)

## Задание 5. Spark SQL: временное представление

Те же данные можно анализировать на чистом SQL.

1. Зарегистрируйте DataFrame `sales` как временное представление с именем `sales`.
2. `top_products` — запрос `spark.sql(...)`: для каждого товара (`product`) — `total_qty` (сумма `qty`) и `revenue` (сумма `qty * unit_price`); 5 товаров с наибольшей выручкой, по убыванию.
3. `tashkent` — запрос по продажам **только в регионе Toshkent**: для каждой пары `category`, `payment` — `orders` (число продаж) и `revenue`; сортировка по убыванию `revenue`.

Выведите оба результата методом `show()`.

<details><summary>💡 Подсказка</summary>

`sales.createOrReplaceTempView("sales")`; запрос: `SELECT product, SUM(qty) AS total_qty, SUM(qty * unit_price) AS revenue FROM sales GROUP BY product ORDER BY revenue DESC LIMIT 5`. Для Ташкента добавьте `WHERE region = 'Toshkent'` и `GROUP BY category, payment`.

</details>

In [ ]:
# Подготовительный код задания
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sales = spark.read.csv("data/spark/sales_events.csv", header=True, inferSchema=True)

In [ ]:
# 1. временное представление
# sales.createOrReplaceTempView(...)

top_products = spark.sql("""
    SELECT product
    FROM sales
""")   # TODO: допишите запрос
top_products.show()

tashkent = ...
# tashkent.show()

## Задание 6. Word count на RDD

Классическая задача Big Data — **подсчёт частот слов** (word count). Решим её для корпуса узбекских новостей `data/corpus_uz/*.txt` на низкоуровневом API RDD.

Функция `tokens(line)` уже написана: она переводит строку в нижний регистр, приводит разные апострофы (ʻ ʼ ‘ ’ и обратный) к `'` и возвращает список слов.

1. `words` — все слова корпуса: `lines.flatMap(tokens)`.
2. `pairs` — пары `(слово, 1)`: `map`.
3. `counts` — частоты: `reduceByKey` складывает единицы для одинаковых ключей.
4. `top10` — **список** из 10 самых частых пар `(слово, частота)` по убыванию частоты (`sortBy(..., ascending=False).take(10)` или `takeOrdered`).

Обратите внимание: `flatMap`, `map` и `reduceByKey` — трансформации, вычисления запускают только действия `count`, `take`, `collect`.

<details><summary>💡 Подсказка</summary>

`words = lines.flatMap(tokens)`, `pairs = words.map(lambda w: (w, 1))`, `counts = pairs.reduceByKey(lambda a, b: a + b)`, `top10 = counts.sortBy(lambda kv: kv[1], ascending=False).take(10)`.

</details>

In [ ]:
import re
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

APOS = re.compile("[\u02bb\u02bc\u2018\u2019\u0060]")   # ʻ ʼ ‘ ’ и обратный апостроф

def tokens(line):
    """Строка -> список слов: нижний регистр, единый апостроф, только латинские буквы."""
    line = APOS.sub("'", line.lower())
    return [w.strip("'") for w in re.split("[^a-z']+", line) if w.strip("'")]

lines = sc.textFile("data/corpus_uz/*.txt")
print("Строк:", lines.count(), "| разделов:", lines.getNumPartitions())

words = ...     # TODO: flatMap
pairs = ...     # TODO: map -> (слово, 1)
counts = ...    # TODO: reduceByKey
top10 = ...     # TODO: 10 самых частых

print("Всего слов:", words.count(), "| уникальных:", counts.count())
for w, n in top10:
    print(f"{w:15} {n}")

## Задание 7. Word count на DataFrame и запись в Parquet

Повторите подсчёт частот средствами DataFrame API и сохраните результат.

1. `spark.read.text(...)` даёт DataFrame с одним столбцом `value` (одна строка файла — одна запись). Стартовый код уже приводит строки к нижнему регистру и унифицирует апострофы.
2. Получите DataFrame `words` со столбцом `word`: разбейте строку `F.split(..., "[^a-z']+")`, разверните массив в строки `F.explode(...)`, уберите апострофы по краям слова (`F.regexp_replace(..., "^'+|'+$", "")`) и отбросьте пустые строки.
3. `freq` — частоты: `groupBy("word").count()`, сортировка по убыванию `count` (при равенстве — по слову).
4. Сохраните `freq` в формате **Parquet** в каталог `output/word_freq` в режиме `overwrite`, затем прочитайте его обратно в DataFrame `back` и сравните число строк.
5. *(По желанию)* Сохраните 20 самых частых слов в CSV с заголовком: `freq.limit(20).write.mode("overwrite").csv("output/top20", header=True)`.

Результаты должны совпасть с RDD-версией из прошлого задания.

<details><summary>💡 Подсказка</summary>

`words = lines.select(F.explode(F.split("line", "[^a-z']+")).alias("word")).select(F.regexp_replace("word", "^'+|'+$", "").alias("word")).filter(F.col("word") != "")`; `freq = words.groupBy("word").count().orderBy(F.desc("count"), "word")`; запись: `freq.write.mode("overwrite").parquet("output/word_freq")`.

</details>

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

text = spark.read.text("data/corpus_uz/*.txt")             # столбец value
APOS = "[\u02bb\u02bc\u2018\u2019\u0060]"

lines = (text
         .select(F.lower(F.col("value")).alias("line"))
         .select(F.regexp_replace("line", APOS, "'").alias("line")))

words = (lines
         # TODO: split -> explode -> alias("word")
         # TODO: убрать апострофы по краям, отбросить пустые слова
        )

freq = ...    # TODO: groupBy("word").count() + сортировка
# freq.show(10)

# TODO: запись в output/word_freq (parquet, overwrite) и чтение обратно в back

## Самостоятельная работа

## Самостоятельная работа: «Роль Apache Spark в области Big Data»

1. Сравните **Hadoop MapReduce** и **Spark** по критериям: модель вычислений, работа с памятью, скорость итеративных алгоритмов, языки, потоковая обработка, удобство. Оформите таблицей.
2. Опишите путь одной программы: `spark-submit` → драйвер → менеджер кластера (YARN) → исполнители → задания, этапы, задачи. Что такое *shuffle* и почему `reduceByKey` создаёт новый этап?
3. Найдите 2–3 примера использования Spark в обработке естественного языка (подготовка корпусов, TF-IDF, Spark NLP) и кратко опишите их.
4. **Практика:** расширьте word count из заданий 6–7 — исключите стоп-слова из `data/stopwords_uz.txt` (не забудьте привести апострофы к одному виду) и найдите 20 самых частых **биграмм**. Сохраните результат в Parquet.